[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/notebooks/showpdf/showpdf_all_features.ipynb)
# ShowPDF — All Features
Comprehensive demo of every ShowPDF capability:

- Simulated amorphous Ta 4D-STEM input (downloaded from Google Drive)
- Probe analysis mode — single-position diffraction average
- Scan-space inclusion mask (paint to add or remove probes)
- The four computed panels: **I(k)**, **F(k)**, **G(r)**, **g(r)**
- Real-space scale bars on the nav panel
- Origin-oscillation damping for cleaner low-r G(r)
- State persistence via `save()` / `state=` parameter
- Tool lock/hide via `disabled_tools` / `hidden_tools`

In [1]:
# Install in Google Colab
try:
    import google.colab
    !pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget
except ImportError:
    pass  # Not in Colab, skip

In [2]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass

env: ANYWIDGET_HMR=1


## 1. Load Data
Simulated amorphous Ta 4D-STEM dataset (20×20 scan, 307×307 detector). Hosted on Google Drive (~120 MB) and downloaded on first run.

In [3]:
import json
from pathlib import Path

import numpy as np

import quantem.widget
from quantem.core.io.serialize import load
from quantem.widget import ShowPDF, profile

DRIVE_ID = "1w7bO1ex0aN6okp-XdKWs_6ECU8kScPUA"
local_path = Path("../../../quantem-tutorials/data/Ta_sim_binned.zip")
cache_path = Path("Ta_sim_binned.zip")

if local_path.exists():
    file_path = local_path
elif cache_path.exists():
    file_path = cache_path
else:
    try:
        import gdown
    except ImportError:
        %pip install -q gdown
        import gdown
    gdown.download(id=DRIVE_ID, output=str(cache_path), quiet=False)
    file_path = cache_path

ds = load(file_path)
# Known q calibration (1/Å per pixel); set on the dataset so the polar
# transform inherits it via ds.sampling[-1] * radial_step.
ds.sampling[-1] = 0.01488

print(f"Shape: {ds.array.shape}, dtype: {ds.array.dtype}")
print(f"q calibration: {ds.sampling[-1]:.5f} Å⁻¹/pix")
print(f"quantem.widget {quantem.widget.__version__}")
profile()

/opt/anaconda3/envs/widget-env/lib/python3.14/site-packages/anywidget/_util.py:283: UserWarning: anywidget: Live-reloading feature is disabled. To enable, please install the 'watchfiles' package.
  start_thread=_should_start_thread(path),


Shape: (20, 20, 307, 307), dtype: float32
q calibration: 0.01488 Å⁻¹/pix
quantem.widget 0.0.14
quantem.widget  0.0.14
quantem         0.1.8
Python          3.14.3 (CPython)
NumPy           2.4.2
PyTorch         2.10.0
Compute         mps (arm)
System RAM      24.0 GB (5.4 GB available)
VRAM            shared (unified memory)
Disk            398.3 GB free / 926.4 GB
Jupyter         Lab 4.5.6
anywidget       0.9.21
Platform        macOS 26.3.1 arm64


## 2. Basic Widget
Default view. The q-axis calibration comes from `ds.sampling[-1]` (set above). Switch between the **I(k)**, **F(k)**, **G(r)**, **g(r)** tabs on the right to see each computed panel.

In [4]:
w_basic = ShowPDF(
    ds,
    title="Ta Simulated PDF",
    k_min_fit=0.05,
    r_max=20.0,
)
w_basic

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 122.00it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [5]:
w_basic.summary()
print(repr(w_basic))

Ta Simulated PDF
════════════════════════════════
Scan:     20 × 20
k range:  [0.00, 2.22] Å⁻¹
Fit:      k=[0.05, 2.22]
Output:   r=[0.00, 20.00], step=0.02
Plot:     Gr
Mask:     full scan (no mask)
ShowPDF(scan=(20, 20), k=[0.1, 2.2])


## 3. Probe Mode
Switch from full-scan averaging to single-probe analysis: the radial mean is computed only over a square region of side `2*probe_size − 1` centred on `(probe_row, probe_col)`. Drag the marker on the scan panel to move the probe live, or set the traits from Python.

In [6]:
w_probe = ShowPDF(
    ds,
    title="Probe demo",
    k_min_fit=0.05,
    r_max=20.0,
    analysis_mode="probe",
    probe_row=10,
    probe_col=10,
    probe_size=3,
)
print(f"Probe centred at ({w_probe.probe_row}, {w_probe.probe_col}), size {w_probe.probe_size} → {w_probe.mask_pixel_count} included probes")
w_probe

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 138.18it/s]


  Estimated rho0 = 0.030506 atoms/A^3
Probe centred at (10, 10), size 3 → 25 included probes


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [7]:
# Move the probe to the upper-left and grow it — both updates re-trigger the analysis.
w_probe.probe_row = 5
w_probe.probe_col = 5
w_probe.probe_size = 5
print(f"Probe now at ({w_probe.probe_row}, {w_probe.probe_col}), size {w_probe.probe_size} → {w_probe.mask_pixel_count} included probes")

  Estimated rho0 = 0.036440 atoms/A^3
  Estimated rho0 = 0.033826 atoms/A^3
  Estimated rho0 = 0.034706 atoms/A^3
Probe now at (5, 5), size 5 → 81 included probes


## 4. Scan-Space Inclusion Mask
The left panel exposes a paint tool with two states. The **+** button is in **add** mode (painting a region adds those probes to the included set), and **−** is in **remove** mode (painting subtracts from the included set). Programmatically, `set_mask(bool_array)` replaces the whole mask; the convention is `True = include`.

In [8]:
# Remove pixels from the included set: start from full and exclude the bottom half.
w_mask_remove = ShowPDF(
    ds,
    title="Mask: remove pixels (start full, drop bottom half)",
    k_min_fit=0.05,
    r_max=20.0,
)
mask = np.ones((w_mask_remove.scan_rows, w_mask_remove.scan_cols), dtype=bool)
mask[w_mask_remove.scan_rows // 2 :, :] = False
w_mask_remove.set_mask(mask)
print(f"Included: {w_mask_remove.mask_pixel_count} / {w_mask_remove.scan_rows * w_mask_remove.scan_cols}")
w_mask_remove

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 116.86it/s]


  Estimated rho0 = 0.035135 atoms/A^3
  Estimated rho0 = 0.035262 atoms/A^3
Included: 200 / 400


ShowPDF(scan=(20, 20), k=[0.1, 2.2], mask=200px)

In [9]:
# Add pixels to the included set: start from empty and include only the top-left quadrant.
w_mask_add = ShowPDF(
    ds,
    title="Mask: add pixels (start empty, include top-left quadrant)",
    k_min_fit=0.05,
    r_max=20.0,
)
mask = np.zeros((w_mask_add.scan_rows, w_mask_add.scan_cols), dtype=bool)
mask[: w_mask_add.scan_rows // 2, : w_mask_add.scan_cols // 2] = True
w_mask_add.set_mask(mask)
print(f"Included: {w_mask_add.mask_pixel_count} / {w_mask_add.scan_rows * w_mask_add.scan_cols}")
w_mask_add

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 136.50it/s]


  Estimated rho0 = 0.035135 atoms/A^3
  Estimated rho0 = 0.034690 atoms/A^3
Included: 100 / 400


ShowPDF(scan=(20, 20), k=[0.1, 2.2], mask=100px)

## 5. Computed Panels
ShowPDF produces four 1-D panels in sequence:
- **I(k)** — radial mean intensity
- **F(k)** — reduced structure factor (background-subtracted, k-weighted, optionally Lorch-windowed)
- **G(r)** — reduced PDF (sine transform of F(k))
- **g(r)** — pair distribution (requires a number density; estimated by default)

You can switch between them via the tabs on the right, or set `plot_mode` at construction.

In [10]:
w_ik = ShowPDF(ds, title="I(k)", k_min_fit=0.05, r_max=20.0, plot_mode="Ik")
w_ik

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 129.89it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [11]:
w_fk = ShowPDF(ds, title="F(k)", k_min_fit=0.05, r_max=20.0, plot_mode="Fk")
w_fk

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 136.50it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [12]:
w_gr = ShowPDF(ds, title="G(r)", k_min_fit=0.05, r_max=20.0, plot_mode="Gr")
w_gr

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 136.21it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [13]:
w_pdf = ShowPDF(ds, title="g(r)", k_min_fit=0.05, r_max=20.0, plot_mode="gr")
w_pdf

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 138.18it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

## 6. Scale Bars
The nav panel auto-detects scan-space calibration from `ds.sampling[0]` and `ds.units[0]`. The Ta simulation only records the **q-space** calibration (`sampling[-1] = 0.01488 Å⁻¹/pix`); the scan step isn't part of the dataset, so the scale bar reads in `px`.

To switch to `Å`, set `ds.sampling[0:2]` and `ds.units[0:2]` before constructing the widget — the auto-detect handles `Å`/`A`/`angstrom` directly and converts `nm` to `Å`. The bar then auto-rounds to a "nice" length and switches to `nm` above 10 Å.

In [14]:
w_scale = ShowPDF(
    ds,
    title="Scale bar (px — no scan calibration in dataset)",
    k_min_fit=0.05,
    r_max=20.0,
)
print(f"Scan calibration: {w_scale.nav_pixel_size} {w_scale.nav_unit}/pix (auto-detected from ds.sampling[0]/units[0])")
w_scale

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 133.19it/s]


  Estimated rho0 = 0.035135 atoms/A^3
Scan calibration: 1.0 px/pix (auto-detected from ds.sampling[0]/units[0])


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

## 7. Origin Damping
Below the first real-space peak, the raw G(r) often shows unphysical oscillations from truncation of the F(k) integration. Setting `damp_origin_oscillations=True` runs an iterative density-aware correction (Yoshimoto–Omote) that suppresses them. `r_cut` is the lower bound for the peak search used to bracket the correction interval.

In [15]:
w_no_damp = ShowPDF(ds, title="No damping", k_min_fit=0.05, r_max=20.0, plot_mode="Gr")
w_no_damp

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 123.02it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

In [16]:
w_damp = ShowPDF(
    ds,
    title="Damping on (Yoshimoto–Omote)",
    k_min_fit=0.05,
    r_max=20.0,
    plot_mode="Gr",
    damp_origin_oscillations=True,
    r_cut=1.0,
)
w_damp

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 137.92it/s]


  Using estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

## 8. State Persistence
Save the full configuration (fit ranges, density choice, plot mode, etc.) to JSON and replay it on a fresh widget. Image data is **not** saved — only display/parameter state, so the JSON is small and shareable.

In [17]:
# Snapshot a deliberately non-default state
w_state = ShowPDF(ds, title="State demo", k_min_fit=0.05, r_max=20.0)
w_state.k_min_fit = 0.05
w_state.k_max_fit = 1.9
w_state.r_max = 6.0
w_state.plot_mode = "gr"

w_state.save("showpdf_state.json")
print("Saved to showpdf_state.json")
print(json.dumps(json.loads(Path("showpdf_state.json").read_text())["state"], indent=2)[:400])

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 110.63it/s]


  Estimated rho0 = 0.035135 atoms/A^3
  Estimated rho0 = 0.030251 atoms/A^3
  Estimated rho0 = 0.030251 atoms/A^3
Saved to showpdf_state.json
{
  "title": "State demo",
  "k_min_fit": 0.05,
  "k_max_fit": 1.9,
  "k_min_window": 0.0,
  "k_max_window": 2.2171199321746826,
  "k_lowpass": 0.0,
  "k_highpass": 0.0,
  "r_min": 0.0,
  "r_max": 6.0,
  "r_step": 0.02,
  "damp_origin_oscillations": false,
  "r_cut": 1.0,
  "density_mode": "estimated",
  "density_value": 0.030250512063503265,
  "analysis_mode": "mask",
  "probe_row": 10,
  "probe_


In [18]:
# Build a fresh widget from a fresh load of the same data and replay the saved state
ds_replay = load(file_path)
ds_replay.sampling[-1] = 0.01488
w_restored = ShowPDF(ds_replay, title="Replayed", state="showpdf_state.json")
w_restored

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 112.41it/s]


  Estimated rho0 = 0.034516 atoms/A^3
  Estimated rho0 = 0.030251 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 1.9])

In [19]:
w_restored.summary()

State demo
════════════════════════════════
Scan:     20 × 20
k range:  [0.00, 2.22] Å⁻¹
Fit:      k=[0.05, 1.90]
Output:   r=[0.00, 6.00], step=0.02
Plot:     gr
Mask:     full scan (no mask)


## 9. Tool Lock / Hide
`disabled_tools` keeps controls visible but non-interactive (good for guided demos). `hidden_tools` removes them entirely. Group keys: `display`, `mask`, `parameters`, `stats`, `export`, `all`.

In [ ]:
w_locked = ShowPDF(
    ds,
    title="Mask locked (read-only demo)",
    k_min_fit=0.05,
    r_max=20.0,
    disabled_tools=["mask"],
)
w_locked

Polar transform: 100%|██████████| 4/4 [00:00<00:00, 105.36it/s]


  Estimated rho0 = 0.035135 atoms/A^3


ShowPDF(scan=(20, 20), k=[0.1, 2.2])

## 10. Cleanup

In [21]:
Path("showpdf_state.json").unlink(missing_ok=True)